In [20]:
# ==========================================================
# FOUNDATIONS OF STATISTICAL INFERENCE & HYPOTHESIS TESTING
# ==========================================================

import numpy as np
import pandas as pd
from scipy.stats import chi2

print("="*90)
print("SECTION 1 : THEORETICAL FUNDAMENTALS")
print("="*90)

print("\nQUESTION 1")
print("-"*60)

print("""
Raw Maximum Likelihood Covariance Estimator

S_ML = (1/n) Σ (Xi - X̄)(Xi - X̄)^T

Expectation:

E[S_ML] = ((n-1)/n) Σ

Since:

(n-1)/n < 1

for finite samples,

E[S_ML] ≠ Σ

Therefore the MLE covariance estimator is biased and
systematically underestimates the true covariance matrix.
""")

print("\nQUESTION 2")
print("-"*60)

print("""
Applying Bessel's correction:

S = (1/(n-1)) Σ (Xi - X̄)(Xi - X̄)^T

Expectation:

E[S] = Σ

Therefore S is the unbiased covariance estimator.
""")

print("\nQUESTION 3")
print("-"*60)

print("""
Type I Error (α)

Reject H0 when H0 is true.

In SHM:
Healthy structure classified as damaged.

------------------------------------------------------------

Type II Error (β)

Fail to reject H0 when H0 is false.

In SHM:
Damaged structure classified as healthy.
""")

print("\nQUESTION 4")
print("-"*60)

print("""
If α becomes extremely small (e.g. 0.001):

• False alarm probability decreases.
• Type II error increases.
• Statistical power decreases.

Power = 1 - β

The healthy-operation confidence ellipsoid expands,
making the monitoring system more conservative.
""")

# ==========================================================
# SLUTSKY THEOREM
# ==========================================================

print("\n")
print("="*90)
print("SECTION 2 : ASYMPTOTIC DISTRIBUTIONS & SLUTSKY THEOREM")
print("="*90)

print("""
Slutsky's Theorem

If

X_n -> X     (in distribution)

and

Y_n -> c     (in probability)

then

X_n + Y_n -> X + c

X_n Y_n -> cX

X_n / Y_n -> X/c
""")

print("\nAPPLICATION")

print("""
Multivariate CLT:

√n (X̄ - μ) -> N(0, Σ)

Since the sample covariance matrix satisfies

S -> Σ

in probability,

Slutsky's theorem allows us to replace
the unknown population covariance Σ
with the empirical covariance S.

Therefore

S^(-1/2) √n (X̄ - μ)

converges to

N(0, I)

which justifies practical Gaussian modeling.
""")

# ==========================================================
# MANOVA NUMERICAL VERIFICATION
# ==========================================================

print("\n")
print("="*90)
print("SECTION 3 : MANOVA FIRST MOMENT HOMOGENEITY TEST")
print("="*90)

# ----------------------------------------------------------
# DATA GENERATOR
# ----------------------------------------------------------

np.random.seed(42)

n_samples = 5000
n_features = 3

base_data = np.random.multivariate_normal(
    mean=[0.5, -0.2, 1.1],
    cov=[
        [0.09, 0.02, 0.01],
        [0.02, 0.06, 0.03],
        [0.01, 0.03, 0.05]
    ],
    size=n_samples
)

# Hidden structural drift
base_data[4000:, 0] += 0.015
base_data[4000:, 2] -= 0.010

df_strain = pd.DataFrame(
    base_data,
    columns=[
        "Strain_Ch1",
        "Strain_Ch2",
        "Strain_Ch3"
    ]
)

# ----------------------------------------------------------
# MANOVA FUNCTION
# ----------------------------------------------------------

def verify_first_moment_homogeneity(
    df,
    g_chunks=5
):

    X = df.values

    n, p = X.shape

    chunk_size = n // g_chunks

    groups = []

    for i in range(g_chunks):

        start = i * chunk_size
        end = (i + 1) * chunk_size

        groups.append(X[start:end])

    global_mean = X.mean(axis=0)

    W = np.zeros((p, p))
    B = np.zeros((p, p))

    for group in groups:

        nk = group.shape[0]

        group_mean = group.mean(axis=0)

        centered = group - group_mean

        W += centered.T @ centered

        diff = (
            group_mean - global_mean
        ).reshape(-1, 1)

        B += nk * (diff @ diff.T)

    wilks_lambda = (
        np.linalg.det(W)
        /
        np.linalg.det(W + B)
    )

    chi_square = -(
        n
        - 1
        - (p + g_chunks) / 2
    ) * np.log(wilks_lambda)

    df_chi = p * (g_chunks - 1)

    p_value = 1 - chi2.cdf(
        chi_square,
        df=df_chi
    )

    return {
        "Wilks_Lambda": wilks_lambda,
        "Bartlett_ChiSquare": chi_square,
        "Degrees_of_Freedom": df_chi,
        "p_value": p_value
    }

# ----------------------------------------------------------
# EXECUTE
# ----------------------------------------------------------

results = verify_first_moment_homogeneity(
    df_strain,
    g_chunks=5
)

print("\nRESULTS")
print("-"*60)

for k, v in results.items():

    print(
        f"{k:<25}: {v}"
    )

# ----------------------------------------------------------
# DECISION
# ----------------------------------------------------------

alpha = 0.05

print("\nFINAL DECISION")
print("-"*60)

if results["p_value"] < alpha:

    print("Reject H0")

    print(
        "The baseline center shifted over time."
    )

    print(
        "First moment homogeneity is violated."
    )

else:

    print("Fail to Reject H0")

    print(
        "No significant evidence of mean drift."
    )

    print(
        "First moment remains homogeneous."
    )

print("="*90)

SECTION 1 : THEORETICAL FUNDAMENTALS

QUESTION 1
------------------------------------------------------------

Raw Maximum Likelihood Covariance Estimator

S_ML = (1/n) Σ (Xi - X̄)(Xi - X̄)^T

Expectation:

E[S_ML] = ((n-1)/n) Σ

Since:

(n-1)/n < 1

for finite samples,

E[S_ML] ≠ Σ

Therefore the MLE covariance estimator is biased and
systematically underestimates the true covariance matrix.


QUESTION 2
------------------------------------------------------------

Applying Bessel's correction:

S = (1/(n-1)) Σ (Xi - X̄)(Xi - X̄)^T

Expectation:

E[S] = Σ

Therefore S is the unbiased covariance estimator.


QUESTION 3
------------------------------------------------------------

Type I Error (α)

Reject H0 when H0 is true.

In SHM:
Healthy structure classified as damaged.

------------------------------------------------------------

Type II Error (β)

Fail to reject H0 when H0 is false.

In SHM:
Damaged structure classified as healthy.


QUESTION 4
-----------------------------------

In [19]:
# ==========================================================
# PCA SUBSPACE OPTIMIZATION ENGINE
# ==========================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

print("="*90)
print("SECTION 4 : PCA SUBSPACE OPTIMIZATION")
print("="*90)

# ----------------------------------------------------------
# SYNTHETIC DATASET
# ----------------------------------------------------------

np.random.seed(42)

n_samples = 2500

# latent factors

f1 = np.random.normal(
    0,
    1,
    n_samples
)

f2 = np.random.normal(
    0,
    1,
    n_samples
)

# sensors

s1 = (
    0.85 * f1
    + 0.10 * f2
    + np.random.normal(0, 0.30, n_samples)
)

s2 = (
    0.80 * f1
    + 0.15 * f2
    + np.random.normal(0, 0.35, n_samples)
)

s3 = (
    0.12 * f1
    + 0.90 * f2
    + np.random.normal(0, 0.25, n_samples)
)

s4 = (
    0.02 * f1
    + 0.05 * f2
    + np.random.normal(0, 1.40, n_samples)
)

df_asset = pd.DataFrame(
    np.vstack(
        [s1, s2, s3, s4]
    ).T,
    columns=[
        "Sensor_1",
        "Sensor_2",
        "Sensor_3",
        "Sensor_4"
    ]
)

# ----------------------------------------------------------
# STANDARDIZATION
# ----------------------------------------------------------

scaler = StandardScaler()

X = scaler.fit_transform(
    df_asset
)

n, p = X.shape

# ----------------------------------------------------------
# COVARIANCE MATRIX
# ----------------------------------------------------------

cov_matrix = np.cov(
    X,
    rowvar=False,
    ddof=1
)

# ----------------------------------------------------------
# EIGEN DECOMPOSITION
# ----------------------------------------------------------

eigvals, eigvecs = np.linalg.eigh(
    cov_matrix
)

idx = np.argsort(
    eigvals
)[::-1]

eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]

# ----------------------------------------------------------
# EXPLAINED VARIANCE
# ----------------------------------------------------------

explained_variance_ratio = (
    eigvals
    /
    eigvals.sum()
)

cumulative_variance = np.cumsum(
    explained_variance_ratio
)

residual_variance = (
    1
    -
    cumulative_variance
)

# ----------------------------------------------------------
# PCA SCORES
# ----------------------------------------------------------

scores = X @ eigvecs

# ----------------------------------------------------------
# T² AND SPE ANALYSIS
# ----------------------------------------------------------

mean_t2 = []
mean_spe = []

for k in range(1, p + 1):

    Pk = eigvecs[:, :k]

    Tk = scores[:, :k]

    lambdas = eigvals[:k]

    # Hotelling T²

    T2 = np.sum(
        (Tk ** 2) / lambdas,
        axis=1
    )

    mean_t2.append(
        np.mean(T2)
    )

    # Reconstruction

    X_hat = Tk @ Pk.T

    residual = X - X_hat

    SPE = np.sum(
        residual ** 2,
        axis=1
    )

    mean_spe.append(
        np.mean(SPE)
    )

# ----------------------------------------------------------
# RESULTS TABLE
# ----------------------------------------------------------

results_df = pd.DataFrame({

    "k":
        np.arange(
            1,
            p + 1
        ),

    "Mean_T2":
        mean_t2,

    "Mean_SPE":
        mean_spe,

    "Explained_Variance_%":
        explained_variance_ratio * 100,

    "Cumulative_Variance_%":
        cumulative_variance * 100,

    "Residual_Variance_%":
        residual_variance * 100

})

print("\nEIGENVALUES")
print("-"*60)

for i, val in enumerate(
    eigvals,
    start=1
):
    print(
        f"PC{i}: {val:.4f}"
    )

print("\n")
print("="*90)
print("PCA OPTIMIZATION RESULTS")
print("="*90)

print(
    results_df.round(4)
)

# ----------------------------------------------------------
# ELBOW DETECTION
# ----------------------------------------------------------

spe_drop = np.diff(
    mean_spe
)

elbow_k = np.argmin(
    spe_drop
) + 2

print("\n")
print("="*90)
print("SUBSPACE INTERPRETATION")
print("="*90)

print(
    f"\nEstimated elbow location: k = {elbow_k}"
)

print(
    f"Mean SPE at k=1 : {mean_spe[0]:.4f}"
)

print(
    f"Mean SPE at k=2 : {mean_spe[1]:.4f}"
)

print(
    "\nThe elbow indicates the approximate"
)

print(
    "hidden dimensionality of the system."
)

print(
    "\nFor this synthetic asset,"
)

print(
    "two latent physical factors were used"
)

print(
    "to generate the sensor array."
)

print(
    "\nTherefore k ≈ 2 is expected."
)

print("="*90)

# ----------------------------------------------------------
# STORE VARIABLES FOR DASHBOARD
# ----------------------------------------------------------

PCA_RESULTS = {

    "eigvals":
        eigvals,

    "eigvecs":
        eigvecs,

    "explained_variance_ratio":
        explained_variance_ratio,

    "cumulative_variance":
        cumulative_variance,

    "residual_variance":
        residual_variance,

    "mean_t2":
        mean_t2,

    "mean_spe":
        mean_spe,

    "scores":
        scores,

    "X":
        X,

    "df_asset":
        df_asset

}

print(
    "\nPCA_RESULTS dictionary created."
)

print(
    "Use this for the dashboard section."
)

print("="*90)

SECTION 4 : PCA SUBSPACE OPTIMIZATION

EIGENVALUES
------------------------------------------------------------
PC1: 1.9752
PC2: 1.0074
PC3: 0.8814
PC4: 0.1376


PCA OPTIMIZATION RESULTS
   k  Mean_T2  Mean_SPE  Explained_Variance_%  Cumulative_Variance_%  \
0  1   0.9996    2.0256               49.3605                49.3605   
1  2   1.9992    1.0186               25.1742                74.5347   
2  3   2.9988    0.1375               22.0272                96.5619   
3  4   3.9984    0.0000                3.4381               100.0000   

   Residual_Variance_%  
0              50.6395  
1              25.4653  
2               3.4381  
3               0.0000  


SUBSPACE INTERPRETATION

Estimated elbow location: k = 2
Mean SPE at k=1 : 2.0256
Mean SPE at k=2 : 1.0186

The elbow indicates the approximate
hidden dimensionality of the system.

For this synthetic asset,
two latent physical factors were used
to generate the sensor array.

Therefore k ≈ 2 is expected.

PCA_RESULTS dictio

In [18]:
# ==========================================================
# FACTOR ANALYSIS ENGINE
# ==========================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FactorAnalysis

print("="*90)
print("SECTION 5 : FACTOR ANALYSIS")
print("="*90)

# ----------------------------------------------------------
# SYNTHETIC DATASET
# ----------------------------------------------------------

np.random.seed(42)

n_samples = 2500

f1 = np.random.normal(
    0,
    1,
    n_samples
)

f2 = np.random.normal(
    0,
    1,
    n_samples
)

s1 = (
    0.85*f1
    +0.10*f2
    +np.random.normal(0,0.30,n_samples)
)

s2 = (
    0.80*f1
    +0.15*f2
    +np.random.normal(0,0.35,n_samples)
)

s3 = (
    0.12*f1
    +0.90*f2
    +np.random.normal(0,0.25,n_samples)
)

s4 = (
    0.02*f1
    +0.05*f2
    +np.random.normal(0,1.40,n_samples)
)

df_asset = pd.DataFrame(
    np.vstack(
        [s1,s2,s3,s4]
    ).T,
    columns=[
        "Sensor_1",
        "Sensor_2",
        "Sensor_3",
        "Sensor_4"
    ]
)

# ----------------------------------------------------------
# STANDARDIZATION
# ----------------------------------------------------------

X = StandardScaler().fit_transform(
    df_asset
)

# ----------------------------------------------------------
# VARIMAX ROTATION
# ----------------------------------------------------------

def varimax(
    Phi,
    gamma=1.0,
    q=50,
    tol=1e-6
):

    p, k = Phi.shape

    R = np.eye(k)

    d = 0

    for i in range(q):

        d_old = d

        Lambda = Phi @ R

        u, s, vh = np.linalg.svd(

            Phi.T @ (
                Lambda**3
                -
                (gamma/p)
                *
                Lambda
                @
                np.diag(
                    np.diag(
                        Lambda.T @ Lambda
                    )
                )
            )
        )

        R = u @ vh

        d = np.sum(s)

        if d_old != 0:

            if d/d_old < 1 + tol:

                break

    return Phi @ R

# ----------------------------------------------------------
# FACTOR ANALYSIS
# ----------------------------------------------------------

n_factors = 2

fa = FactorAnalysis(
    n_components=n_factors,
    random_state=42
)

fa.fit(X)

# Raw loadings

loadings = fa.components_.T

# Rotated loadings

rot_loadings = varimax(
    loadings
)

# ----------------------------------------------------------
# COMMUNALITIES
# ----------------------------------------------------------

communalities = np.sum(
    rot_loadings**2,
    axis=1
)

# ----------------------------------------------------------
# UNIQUENESS
# ----------------------------------------------------------

uniqueness = (
    1
    -
    communalities
)

uniqueness = np.clip(
    uniqueness,
    0,
    None
)

# ----------------------------------------------------------
# THOMSON REGRESSION SCORES
# ----------------------------------------------------------

Psi = np.diag(
    uniqueness
)

L = rot_loadings

score_weights = np.linalg.inv(
    L.T
    @
    np.linalg.inv(Psi)
    @
    L
) @ (
    L.T
    @
    np.linalg.inv(Psi)
)

factor_scores = (
    X
    @
    score_weights.T
)

# ----------------------------------------------------------
# FACTOR VARIANCES
# ----------------------------------------------------------

factor_variances = np.var(
    factor_scores,
    axis=0,
    ddof=1
)

# ----------------------------------------------------------
# LOADINGS TABLE
# ----------------------------------------------------------

loadings_df = pd.DataFrame(

    rot_loadings,

    index=df_asset.columns,

    columns=[
        "Factor_1",
        "Factor_2"
    ]
)

# ----------------------------------------------------------
# COMMUNALITY TABLE
# ----------------------------------------------------------

comm_df = pd.DataFrame({

    "Communality_h2":
        communalities,

    "Uniqueness_phi2":
        uniqueness

},

index=df_asset.columns)

# ----------------------------------------------------------
# FACTOR VARIANCE TABLE
# ----------------------------------------------------------

factor_var_df = pd.DataFrame({

    "Factor":
        [
            "Factor_1",
            "Factor_2"
        ],

    "Variance":
        factor_variances

})

# ----------------------------------------------------------
# DISPLAY
# ----------------------------------------------------------

print("\nROTATED LOADINGS")
print("-"*60)

print(
    loadings_df.round(4)
)

print("\n")
print("="*90)
print("COMMUNALITY / UNIQUENESS")
print("="*90)

print(
    comm_df.round(4)
)

print("\n")
print("="*90)
print("FACTOR SCORE VARIANCES")
print("="*90)

print(
    factor_var_df.round(4)
)

# ----------------------------------------------------------
# SENSOR DIAGNOSTICS
# ----------------------------------------------------------

highest_noise_sensor = comm_df[
    "Uniqueness_phi2"
].idxmax()

highest_noise_value = comm_df[
    "Uniqueness_phi2"
].max()

print("\n")
print("="*90)
print("SENSOR NOISE ANALYSIS")
print("="*90)

print(
    f"Highest uniqueness sensor : "
    f"{highest_noise_sensor}"
)

print(
    f"Uniqueness value : "
    f"{highest_noise_value:.4f}"
)

print(
    "\nInterpretation:"
)

print(
    "Large uniqueness indicates"
)

print(
    "sensor-specific noise"
)

print(
    "rather than shared"
)

print(
    "structural information."
)

# ----------------------------------------------------------
# STORE VARIABLES
# ----------------------------------------------------------

FA_RESULTS = {

    "X":
        X,

    "loadings":
        rot_loadings,

    "communalities":
        communalities,

    "uniqueness":
        uniqueness,

    "factor_scores":
        factor_scores,

    "factor_variances":
        factor_variances,

    "df_asset":
        df_asset

}

print("\n")
print("="*90)
print("FA_RESULTS dictionary created.")
print("Ready for dashboard generation.")
print("="*90)

SECTION 5 : FACTOR ANALYSIS

ROTATED LOADINGS
------------------------------------------------------------
          Factor_1  Factor_2
Sensor_1    0.9300    0.0040
Sensor_2    0.9246   -0.1436
Sensor_3    0.2129   -0.4824
Sensor_4    0.0339   -0.0867


COMMUNALITY / UNIQUENESS
          Communality_h2  Uniqueness_phi2
Sensor_1          0.8649           0.1351
Sensor_2          0.8756           0.1244
Sensor_3          0.2780           0.7220
Sensor_4          0.0087           0.9913


FACTOR SCORE VARIANCES
     Factor  Variance
0  Factor_1    1.0957
1  Factor_2    3.5678


SENSOR NOISE ANALYSIS
Highest uniqueness sensor : Sensor_4
Uniqueness value : 0.9913

Interpretation:
Large uniqueness indicates
sensor-specific noise
rather than shared
structural information.


FA_RESULTS dictionary created.
Ready for dashboard generation.


In [17]:
# ==========================================================
# PCA OPTIMIZATION DASHBOARD
# ==========================================================

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

eigvals = PCA_RESULTS["eigvals"]
eigvecs = PCA_RESULTS["eigvecs"]

explained = PCA_RESULTS["explained_variance_ratio"]*100
cumulative = PCA_RESULTS["cumulative_variance"]*100
residual = PCA_RESULTS["residual_variance"]*100

mean_t2 = PCA_RESULTS["mean_t2"]
mean_spe = PCA_RESULTS["mean_spe"]

features = PCA_RESULTS["df_asset"].columns

fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=[
        "PCA Loadings Matrix",
        "Eigenvalues",
        "Explained Variance",
        "Residual Variance",
        "Mean Hotelling T²",
        "Mean SPE"
    ]
)

# --------------------------------------------------
# Heatmap
# --------------------------------------------------

fig.add_trace(

    go.Heatmap(
        z=np.abs(eigvecs),
        x=[f"PC{i+1}" for i in range(len(eigvals))],
        y=features,
        colorscale="YlOrRd"
    ),

    row=1,
    col=1
)

# --------------------------------------------------
# Eigenvalues
# --------------------------------------------------

fig.add_trace(

    go.Bar(
        x=[f"PC{i+1}" for i in range(len(eigvals))],
        y=eigvals,
        name="Eigenvalues"
    ),

    row=1,
    col=2
)

# --------------------------------------------------
# Explained Variance
# --------------------------------------------------

fig.add_trace(

    go.Bar(
        x=[f"PC{i+1}" for i in range(len(eigvals))],
        y=explained,
        name="Explained %"
    ),

    row=1,
    col=3
)

fig.add_trace(

    go.Scatter(
        x=[f"PC{i+1}" for i in range(len(eigvals))],
        y=cumulative,
        mode="lines+markers",
        name="Cumulative %"
    ),

    row=1,
    col=3
)

# --------------------------------------------------
# Residual Variance
# --------------------------------------------------

fig.add_trace(

    go.Bar(
        x=np.arange(1,len(eigvals)+1),
        y=residual,
        name="Residual %"
    ),

    row=2,
    col=1
)

# --------------------------------------------------
# Mean T²
# --------------------------------------------------

fig.add_trace(

    go.Scatter(
        x=np.arange(1,len(mean_t2)+1),
        y=mean_t2,
        mode="lines+markers",
        name="Mean T²"
    ),

    row=2,
    col=2
)

# --------------------------------------------------
# Mean SPE
# --------------------------------------------------

fig.add_trace(

    go.Scatter(
        x=np.arange(1,len(mean_spe)+1),
        y=mean_spe,
        mode="lines+markers",
        name="Mean SPE"
    ),

    row=2,
    col=3
)

fig.update_layout(

    height=750,
    width=1400,
    template="plotly_white",
    title="PCA OPTIMIZATION SUITE",

    showlegend=False
)

fig.show()

In [16]:
# ==========================================================
# FACTOR ANALYSIS DASHBOARD
# ==========================================================

import plotly.graph_objects as go
from plotly.subplots import make_subplots

loadings = FA_RESULTS["loadings"]

communalities = FA_RESULTS["communalities"]
uniqueness = FA_RESULTS["uniqueness"]

factor_variances = FA_RESULTS["factor_variances"]

sensor_names = FA_RESULTS["df_asset"].columns

fig = make_subplots(

    rows=2,
    cols=2,

    horizontal_spacing=0.24,
    vertical_spacing=0.28,

    subplot_titles=[

        "Structural Loadings Heatmap",

        "Communality vs Uniqueness",

        "Sensor Noise Floor",

        "Factor Variance"
    ]
)

# --------------------------------------------------
# Heatmap
# --------------------------------------------------

fig.add_trace(

    go.Heatmap(

        z=np.abs(loadings),

        x=[
            "Factor 1",
            "Factor 2"
        ],

        y=sensor_names,

        colorscale="YlOrRd",

        colorbar=dict(

            x=-0.15,

            len=0.38,

            y=0.78,

            yanchor="middle",

            xanchor="right"
        )
    ),

    row=1,
    col=1
)

# --------------------------------------------------
# Stacked Bars
# --------------------------------------------------

fig.add_trace(

    go.Bar(

        y=sensor_names,

        x=communalities*100,

        orientation="h",

        name="Communality",

        marker_color="#1f77b4"
    ),

    row=1,
    col=2
)

fig.add_trace(

    go.Bar(

        y=sensor_names,

        x=uniqueness*100,

        orientation="h",

        name="Uniqueness",

        marker_color="#ff7f0e"
    ),

    row=1,
    col=2
)

# --------------------------------------------------
# Noise Floor
# --------------------------------------------------

fig.add_trace(

    go.Scatter(

        x=sensor_names,

        y=uniqueness,

        mode="lines+markers",

        line=dict(

            color="#d62728",

            width=2,

            dash="dashdot"
        ),

        marker=dict(

            symbol="x"
        ),

        name="Uniqueness"
    ),

    row=2,
    col=1
)

# --------------------------------------------------
# Factor Variance
# --------------------------------------------------

fig.add_trace(

    go.Bar(

        x=[
            "Factor 1",
            "Factor 2"
        ],

        y=factor_variances,

        marker_color="#2ca02c",

        marker_line_color="black",

        marker_line_width=0.5
    ),

    row=2,
    col=2
)

# --------------------------------------------------
# Layout
# --------------------------------------------------

fig.update_xaxes(

    range=[0,100],

    row=1,
    col=2
)

fig.update_xaxes(

    tickangle=25,

    row=2,
    col=1
)

fig.update_layout(

    width=1250,

    height=750,

    template="plotly_white",

    barmode="stack",

    legend=dict(

        orientation="h",

        x=0.5,

        y=1.02,

        xanchor="center"
    ),

    margin=dict(

        t=150,

        b=60,

        l=140,

        r=80
    ),

    title="FA LATENT SUBSPACE DIAGNOSTICS"
)

fig.show()

Question 1

In [12]:
print("="*80)
print("QUESTION 1 - TOTAL VARIANCE ILLUSION")
print("="*80)

# from FA results (if not run yet, use interpretation form)
try:
    uniq = FA_RESULTS["uniqueness"]
    sensors = FA_RESULTS["df_asset"].columns

    print("\nSensor Uniqueness Values:")
    for s, u in zip(sensors, uniq):
        print(f"{s}: {u:.4f}")

    max_idx = np.argmax(uniq)

    print("\nINTERPRETATION:")
    print(f"{sensors[max_idx]} has highest uniqueness ({uniq[max_idx]:.4f})")

except:
    print("""
Interpretation (generic if FA not executed):

Sensor_4 shows very high uniqueness (~1),
meaning it is dominated by sensor-specific noise
rather than shared structural signal.

PCA mistake:
It treats this high variance noise as important structure,
inflating top eigenvalues and misleading the model.
""")

print("\nFINAL ANSWER:")
print("""
Sensor_4 is a noise-dominated channel.
PCA is misled because it maximizes total variance,
so large sensor noise inflates principal components,
creating a false impression of strong structure.

Risk: False anomaly patterns and incorrect shutdown decisions.
""")

QUESTION 1 - TOTAL VARIANCE ILLUSION

Sensor Uniqueness Values:
Sensor_1: 0.1351
Sensor_2: 0.1244
Sensor_3: 0.7220
Sensor_4: 0.9913

INTERPRETATION:
Sensor_4 has highest uniqueness (0.9913)

FINAL ANSWER:

Sensor_4 is a noise-dominated channel.
PCA is misled because it maximizes total variance,
so large sensor noise inflates principal components,
creating a false impression of strong structure.

Risk: False anomaly patterns and incorrect shutdown decisions.



Question 2

In [13]:
print("="*80)
print("QUESTION 2 - VARIMAX ROTATION EFFECT")
print("="*80)

print("""
PCA enforces global variance maximization,
so loadings are mixed across sensors.

Varimax rotation changes the coordinate system
without changing total explained variance.

Effect:
- Makes large loadings larger
- Makes small loadings smaller
- Produces "simple structure"

RESULT:
Sensor_1 + Sensor_2 → Factor 1
Sensor_3 → Factor 2
Sensor_4 → Noise factor (weak loading)

WHY BETTER:
Operators can directly map factors to physical subsystems,
making fault localization much easier than PCA eigenvectors.
""")

QUESTION 2 - VARIMAX ROTATION EFFECT

PCA enforces global variance maximization,
so loadings are mixed across sensors.

Varimax rotation changes the coordinate system
without changing total explained variance.

Effect:
- Makes large loadings larger
- Makes small loadings smaller
- Produces "simple structure"

RESULT:
Sensor_1 + Sensor_2 → Factor 1
Sensor_3 → Factor 2
Sensor_4 → Noise factor (weak loading)

WHY BETTER:
Operators can directly map factors to physical subsystems,
making fault localization much easier than PCA eigenvectors.



Question 3

In [14]:
print("="*80)
print("QUESTION 3 - PCA SUBSPACE TRUNCATION (T2 & SPE)")
print("="*80)

try:
    spe = PCA_RESULTS["mean_spe"]

    k_elbow = np.argmin(np.diff(spe)) + 2

    print(f"Mean SPE values: {np.round(spe,4)}")
    print(f"\nEstimated elbow k ≈ {k_elbow}")

    print("""
INTERPRETATION:

As k increases:
- SPE decreases sharply at first (signal captured)
- Then flattens (only noise remains)

The elbow indicates true latent dimensionality.

For this system:
→ 2 latent physical factors exist
→ so k ≈ 2 is optimal

If k is too high:
Noise enters principal subspace → reduces fault sensitivity
""")

except:
    print("""
Generic Interpretation:

SPE drops sharply at first,
then becomes flat after true latent dimension is captured.

Elbow ≈ 2 indicates system has 2 physical driving factors.

Choosing higher k forces noise into "clean" subspace.
""")

QUESTION 3 - PCA SUBSPACE TRUNCATION (T2 & SPE)
Mean SPE values: [2.0256 1.0186 0.1375 0.    ]

Estimated elbow k ≈ 2

INTERPRETATION:

As k increases:
- SPE decreases sharply at first (signal captured)
- Then flattens (only noise remains)

The elbow indicates true latent dimensionality.

For this system:
→ 2 latent physical factors exist
→ so k ≈ 2 is optimal

If k is too high:
Noise enters principal subspace → reduces fault sensitivity



Question 4

In [15]:
print("="*80)
print("QUESTION 4 - PCA vs FA ROBUSTNESS")
print("="*80)

try:
    uniq = FA_RESULTS["uniqueness"]
    sensors = FA_RESULTS["df_asset"].columns

    print("Sensor Uniqueness Values:")
    for s, u in zip(sensors, uniq):
        print(f"{s}: {u:.4f}")

    worst = sensors[np.argmax(uniq)]
    worst_val = np.max(uniq)

    print(f"\nMost noisy sensor: {worst} ({worst_val:.4f})")

except:
    print("""
Generic interpretation:

Sensor_4 has highest uniqueness → pure noise source.

""")

print("""
FINAL COMPARISON:

PCA Strategy:
- Uses total variance
- Sensitive to noisy sensors
- Faults can distort principal components

FA Strategy:
- Separates shared vs unique variance
- Uses uniqueness (φ²) to isolate noise
- Robust against single sensor failure

FINAL ANSWER:
Factor Analysis is more robust for sensor failure
because uniqueness explicitly isolates sensor-specific noise,
preventing it from contaminating system-wide diagnostics.
""")

QUESTION 4 - PCA vs FA ROBUSTNESS
Sensor Uniqueness Values:
Sensor_1: 0.1351
Sensor_2: 0.1244
Sensor_3: 0.7220
Sensor_4: 0.9913

Most noisy sensor: Sensor_4 (0.9913)

FINAL COMPARISON:

PCA Strategy:
- Uses total variance
- Sensitive to noisy sensors
- Faults can distort principal components

FA Strategy:
- Separates shared vs unique variance
- Uses uniqueness (φ²) to isolate noise
- Robust against single sensor failure

FINAL ANSWER:
Factor Analysis is more robust for sensor failure
because uniqueness explicitly isolates sensor-specific noise,
preventing it from contaminating system-wide diagnostics.

